<a href="https://colab.research.google.com/github/lilliekate/lilliekate/blob/main/CFB_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#----
## 1) INSTALL MINIMAL DEPENDENCIES
#----
# We are only installing packages here. YOU MUST RESTART THE KERNEL AFTER THIS CELL RUNS.
!pip install requests beautifulsoup4 python-dateutil pandas tqdm ratelimit tenacity --upgrade
print('✅ Dependencies installation finished.')

# Do not run the next cell until you have restarted the session.

✅ Dependencies installation finished.


In [ ]:
# ---
## 2) Imports & Constants
# ---

import os, re, json, time, math, hashlib
from datetime import datetime, timedelta, timezone, date
from dateutil import parser as dateparser
from typing import List, Dict, Optional

import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
from ratelimit import limits, sleep_and_retry
from tenacity import retry, stop_after_attempt, wait_exponential

BASE = 'https://www.cbssports.com'
# Using the index for broad discovery
NEWS_SITEMAP_INDEX = 'https://www.cbssports.com/sitemaps/all-news-sitemap.xml'
USER_AGENT = (
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 '
    '(KHTML, like Gecko) Chrome/118.0.0.0 Safari/537.36'
)
HEADERS = {'User-Agent': USER_AGENT, 'Accept-Language': 'en-US,en;q=0.9'}

# --- GLOBAL CONFIGURATION (Overall Project Range) ---
OVERALL_START_DATE = '2023-11-05'   # inclusive
OVERALL_END_DATE   = '2025-11-05'   # inclusive

# RUNTIME VARIABLES (These will be overwritten by Cell 4)
START_DATE = OVERALL_START_DATE
END_DATE   = OVERALL_END_DATE
# -----------------------------------------------------

PREGAME_WINDOW_DAYS = 14    # used later in Phase 2
REQUESTS_PER_10S = 10       # polite rate limit
# OUTPUT_DIR = 'data' # REMOVED: Using PROJECT_DIR from Cell 3.5 instead

# os.makedirs(OUTPUT_DIR, exist_ok=True) # REMOVED: Managed in Cell 3.5
print('✅ Imports & config ready.')

✅ Imports & config ready.


In [ ]:
# ---
## 3) Networking Helpers
# ---

def iso(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).isoformat()

@sleep_and_retry
@limits(calls=REQUESTS_PER_10S, period=10)
def get(url: str, **kwargs) -> requests.Response:
    resp = requests.get(url, headers=HEADERS, timeout=20, **kwargs)
    resp.raise_for_status()
    return resp

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=8))
def safe_get(url: str, **kwargs) -> Optional[requests.Response]:
    return get(url, **kwargs)

print('✅ Net utils ready.')

✅ Net utils ready.


Run 2nd:

In [ ]:
# ---
## 3.5) Mount & Load Google Drive
# ---

# FIX: Force-reinstall pyarrow and pandas to resolve binary compatibility warnings.
# --no-cache-dir ensures the packages are re-downloaded and re-linked against the
# current environment's core libraries.
!pip install -q --no-cache-dir pyarrow==16.1.0 pandas

from google.colab import drive
import os
import pandas as pd

# Define the persistent folder path inside your Google Drive for all project data
PROJECT_DIR = '/content/drive/MyDrive/CFB_Project_Data_v3'

# Command to mount Google Drive
drive.mount('/content/drive')
print('\n--- Mount Status ---')

# Check if the project folder exists, and create it if it doesn't
if not os.path.exists(PROJECT_DIR):
    os.makedirs(PROJECT_DIR)
    print(f'✅ Created project directory: {PROJECT_DIR}')
else:
    print(f'✅ Found project directory: {PROJECT_DIR}')

print('--------------------')

# CRITICAL STEP: After running this cell, you MUST restart the session
# for the newly installed packages to take effect and eliminate the warnings.
# Go to Runtime -> Restart session.

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- Mount Status ---
✅ Found project directory: /content/drive/MyDrive/CFB_Project_Data_v3
--------------------


In [ ]:
## Restart session here!

In [ ]:
# ---
## 4) MANUAL SEGMENTATION CONTROL (Must run this cell first!)
# ---
# Explicitly import typing components in case Cell 2 wasn't run first
from typing import List, Dict
from datetime import timedelta
from dateutil import parser as dateparser # Using the same import name as Cell 2
from datetime import date # Added for date objects

# --------------------------------------------------------------------------
# 🛑 ACTION REQUIRED: SET THE INDEX 🛑
# 1. Choose which segment to run next by setting the index (starts at 0).
#    - Index 0: First 4-month chunk
#    - Index 1: Second 4-month chunk, and so on.
# 2. Run this cell, then proceed to run Cell 5 (Discovery) and Cell 6 (Fetching).
# 3. Change the index and repeat the process until all segments are complete.
# --------------------------------------------------------------------------

# Set the index of the segment you want to run right now (starts at 0)
CURRENT_SEGMENT_INDEX = 0 # <--- EDIT THIS INDEX FOR EACH 4-MONTH RUN

def generate_4_month_segments(start_date_str: str, end_date_str: str) -> List[Dict]:
    """Generates a list of ~4-month segments between the overall start and end dates."""
    # Ensure dateparser is defined and used correctly
    overall_start = dateparser.parse(start_date_str).date()
    overall_end = dateparser.parse(end_date_str).date()

    segments = []
    current_start = overall_start

    while current_start <= overall_end:
        # Calculate ~4 months (120 days) from the current start date
        current_end_candidate = current_start + timedelta(days=120)

        # Ensure the end date doesn't exceed the overall project end date
        current_end = min(current_end_candidate, overall_end)

        segments.append({
            'index': len(segments),
            'start_date': current_start.isoformat(),
            'end_date': current_end.isoformat(),
        })

        # Move the start date to the day after the current end date
        current_start = current_end + timedelta(days=1)

        # Stop if we've processed all dates up to the overall end
        if current_start > overall_end:
            break

    return segments

# --- Run Segmentation ---
# NOTE: OVERALL_START_DATE and OVERALL_END_DATE are defined in Cell 2.
SEGMENTS = generate_4_month_segments(OVERALL_START_DATE, OVERALL_END_DATE)

# Check if the requested index is valid
if not SEGMENTS or CURRENT_SEGMENT_INDEX >= len(SEGMENTS):
    raise IndexError(f"ERROR: CURRENT_SEGMENT_INDEX ({CURRENT_SEGMENT_INDEX}) is out of range. Max index is {len(SEGMENTS) - 1}")

# Set the global runtime variables for the current segment
SEGMENT_INFO = SEGMENTS[CURRENT_SEGMENT_INDEX]
START_DATE = SEGMENT_INFO['start_date']
END_DATE = SEGMENT_INFO['end_date']

if 0 <= CURRENT_SEGMENT_INDEX < len(SEGMENTS):
    selected_segment = SEGMENTS[CURRENT_SEGMENT_INDEX]
    # OVERWRITE GLOBAL DATES for the next cells (5 and 6)
    START_DATE = selected_segment['start_date']
    END_DATE   = selected_segment['end_date']

    print("--- SEGMENTATION REPORT ---")
    print(f"Overall Project Range: {OVERALL_START_DATE} to {OVERALL_END_DATE}")
    print(f"Total Segments Found: {len(SEGMENTS)}")
    print(f"Current Segment Index: {CURRENT_SEGMENT_INDEX}\n")

    # Display all segments for context
    for i, seg in enumerate(SEGMENTS):
        status = "<-- NEXT RUN" if i == CURRENT_SEGMENT_INDEX else ""
        print(f"Segment {i}: {seg['start_date']} to {seg['end_date']} {status}")

    print(f"\n✅ OVERWRITING RUN DATES: {START_DATE} to {END_DATE}. Now run Cell 5 and Cell 6.")
else:
    print(f"❌ ERROR: Invalid segment index {CURRENT_SEGMENT_INDEX}. Must be between 0 and {len(SEGMENTS) - 1}.")
    print(f"Using original full range: {OVERALL_START_DATE} to {OVERALL_END_DATE}.")

--- SEGMENTATION REPORT ---
Overall Project Range: 2023-11-05 to 2025-11-05
Total Segments Found: 7
Current Segment Index: 0

Segment 0: 2023-11-05 to 2024-03-04 <-- NEXT RUN
Segment 1: 2024-03-05 to 2024-07-03 
Segment 2: 2024-07-04 to 2024-11-01 
Segment 3: 2024-11-02 to 2025-03-02 
Segment 4: 2025-03-03 to 2025-07-01 
Segment 5: 2025-07-02 to 2025-10-30 
Segment 6: 2025-10-31 to 2025-11-05 

✅ OVERWRITING RUN DATES: 2023-11-05 to 2024-03-04. Now run Cell 5 and Cell 6.


In [ ]:
# ---
## 5) Safe Discovery (Sitemap Index) with Smart Time Caching
# THIS CELL NOW RUNS FOR THE DATE RANGE SET IN CELL 4
# ---

# 1. Define Unique Cache Path based on current START/END dates
# We use PROJECT_DIR (from Cell 3.5) instead of OUTPUT_DIR
cache_filename = f"discovered_urls_{START_DATE.replace('-', '')}_{END_DATE.replace('-', '')}.parquet"
DISCOVERY_CACHE_PATH = os.path.join(PROJECT_DIR, cache_filename)

def to_utc(dt):
    # Ensure datetime object is timezone-aware and in UTC
    return dt.replace(tzinfo=timezone.utc) if dt.tzinfo is None else dt.astimezone(timezone.utc)

start_dt = to_utc(dateparser.parse(START_DATE))
end_dt   = to_utc(dateparser.parse(END_DATE)) + timedelta(days=1) - timedelta(seconds=1)

def fetch_xml(url):
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    return BeautifulSoup(r.text, 'xml')

# --- Discovery Function (No Change) ---
def discover_from_index_v3(start, end, limit_sitemaps=200):
    try:
        idx = fetch_xml(NEWS_SITEMAP_INDEX)
    except Exception as e:
        print(f"❌ CRITICAL ERROR: Could not fetch sitemap index: {e}")
        return []

    site_urls = [loc.text.strip() for loc in idx.find_all('loc')]
    discovered_rows = []

    for sitemap_url in tqdm(site_urls[:limit_sitemaps], desc="monthly sitemaps"):
        try:
            sm = fetch_xml(sitemap_url)
        except Exception as e:
            print(f"❌ Failed to fetch sitemap {sitemap_url}: {e}")
            continue

        for url in sm.find_all('url'):
            loc_tag = url.find('loc')
            pub_tag = url.find('news:publication_date')
            if not loc_tag or not pub_tag:
                continue

            loc = (loc_tag.text or '').strip()

            try:
                pub_dt = to_utc(dateparser.parse(pub_tag.text.strip()))
            except Exception:
                continue

            if not (start <= pub_dt <= end):
                continue

            if f"{BASE}/college-football/" not in loc:
                continue

            discovered_rows.append({'url': loc, 'published': pub_dt})

        time.sleep(0.2)

    seen, uniq = set(), []
    for r in discovered_rows:
        if r['url'] not in seen:
            seen.add(r['url'])
            uniq.append(r)
    return uniq
# --- End Discovery Function ---


# --- Caching Logic (MODIFIED to use Drive path) ---
if os.path.exists(DISCOVERY_CACHE_PATH):
    print(f"✅ Loading discovered URLs for {START_DATE} to {END_DATE} from cache: {DISCOVERY_CACHE_PATH}")
    df_discovered = pd.read_parquet(DISCOVERY_CACHE_PATH)
    discovered = df_discovered.to_dict('records')

else:
    print(f"⏳ Cache file not found for this date range ({START_DATE} to {END_DATE}). Starting long discovery process...")
    discovered = discover_from_index_v3(start_dt, end_dt)

    # Save the discovered list for next time
    if discovered:
        df_discovered = pd.DataFrame(discovered)
        # Ensure timestamp column is correct for Parquet
        df_discovered['published'] = df_discovered['published'].astype('datetime64[ns, UTC]')
        df_discovered.to_parquet(DISCOVERY_CACHE_PATH, index=False)
        print(f"💾 Discovered {len(discovered)} URLs and saved to cache: {DISCOVERY_CACHE_PATH}.")
    else:
        print("⚠️ No URLs discovered to save to cache.")


print(f"✅ Found {len(discovered)} CBS college-football URLs in your window.")
print("Example:", discovered[0] if discovered else "None")

⏳ Cache file not found for this date range (2023-11-05 to 2024-03-04). Starting long discovery process...


monthly sitemaps:  46%|████▋     | 33/71 [09:43<11:12, 17.69s/it]


KeyboardInterrupt: 

In [ ]:
# ---
## 6) Article Fetching & Parsing with Incremental Caching
# THIS CELL NOW RUNS FOR THE DATE RANGE SET IN CELL 4, AND WILL APPEND TO THE GLOBAL CACHE.
# ---

# Define the cache path (Using PROJECT_DIR from Cell 3.5)
FETCHED_CACHE_PATH = os.path.join(PROJECT_DIR, 'articles_raw.parquet')

def extract_main_text(html: str) -> str:
    # ... (function body remains the same)
    soup = BeautifulSoup(html, 'html.parser')
    for tag in soup(['script','style','noscript','header','footer','aside','nav']):
        tag.decompose()
    title = None
    if soup.title and soup.title.string:
        title = soup.title.string.strip()
    og_title = soup.find('meta', attrs={'property': 'og:title'})
    if og_title and og_title.get('content'):
        title = og_title['content'].strip()
    paras = [p.get_text(' ', strip=True) for p in soup.find_all('p') if p.get_text(strip=True)]
    text = '\n'.join(paras)
    return title or '', text

def fetch_articles(rows_to_fetch: List[Dict]) -> List[Dict]:
    # This function now only handles the network calls for a subset of rows
    out = []
    for row in tqdm(rows_to_fetch, desc='fetching NEW articles'):
        url = row['url']
        try:
            resp = safe_get(url)
            if not resp or not resp.ok:
                continue

            title, text = extract_main_text(resp.text)

            if not text:
                print(f"⚠️ Warning: No text extracted for {url}")
                continue

            out.append({
                'url': url,
                'published_utc': row['published'].astimezone(timezone.utc).isoformat(),
                'title': title,
                'article_text': text,
                'text_hash': hashlib.sha256(text.encode('utf-8', errors='ignore')).hexdigest(),
            })
            # time.sleep(0.5) # Commented out for speed
        except Exception as e:
            print(f"❌ Failed to process article {url}: {e}")
            continue
    return out

# --- Caching Logic (MODIFIED to use Drive path) ---
if os.path.exists(FETCHED_CACHE_PATH):
    # 1. Load existing cache
    df_cached = pd.read_parquet(FETCHED_CACHE_PATH)
    cached_urls = set(df_cached['url'])

    # 2. Identify new URLs by comparing Cell 5 results (discovered) with the global cache (df_cached)
    urls_to_fetch = [r for r in discovered if r['url'] not in cached_urls]

    print(f"✅ Cache found. Total discovered URLs for this segment: {len(discovered)}. Cached articles (Total): {len(df_cached)}. Fetching new articles: {len(urls_to_fetch)}")

    # 3. Fetch only the new articles
    newly_fetched = fetch_articles(urls_to_fetch)

    # 4. Combine old and new data
    df_newly_fetched = pd.DataFrame(newly_fetched)
    df_combined = pd.concat([df_cached, df_newly_fetched], ignore_index=True)

    # Update the global articles list/DataFrame
    articles = df_combined.to_dict('records')
    # Save the complete, combined set back to the cache (overwriting the old one)
    df_combined.to_parquet(FETCHED_CACHE_PATH, index=False)
    print(f"💾 Combined and saved {len(df_combined)} articles to cache: {FETCHED_CACHE_PATH}.")

else:
    # No cache exists, run full fetch (only for this segment, which starts the cache)
    print("⏳ Article cache not found. Starting full article fetching process...")
    articles = fetch_articles(discovered)

    if articles:
        # Save results immediately to start the cache
        df_articles_full = pd.DataFrame(articles)
        df_articles_full.to_parquet(FETCHED_CACHE_PATH, index=False)
        print(f"💾 Saved {len(articles)} articles to new cache: {FETCHED_CACHE_PATH}.")


print(f"✅ Successfully processed {len(articles)} articles (newly fetched + cached).")

In [ ]:
# ---
## 7) Trash-talk Detection
# ---

TRASH_KEYWORDS = [
    r"\btrash talk\b",
    r"\btrash-talk\b",
    r"\btrash-talking\b",
    r"\bbulletin[- ]board material\b",
    r"\bcalled (?:him|her|them|us) out\b",
    r"\bguarantee(?:d|ing)? (?:a )?win\b",
    r"\bwe'?re going to (?:destroy|crush|beat|embarrass)\b",
    r"\bthey'?re (?:soft|overrated|not good|weak)\b",
    r"\btook a shot at\b",
    r"\bfired shots?\b",
    r"\bclap(?:ping)? back\b",
    r"\bwar of words\b",
    r"\bverbal jab\b",
    r"\bfeud\b",
    r"\bbeef\b",
]
TRASH_REGEX = re.compile("|".join(TRASH_KEYWORDS), flags=re.IGNORECASE)

# Simple quote capture — returns any quoted strings in a line
QUOTE_REGEX = re.compile(r'([“”"\'])(.+?)\1')

def detect_trash_talk(text: str) -> Dict:
    hits = []
    lines = [ln.strip() for ln in text.split('\n') if ln.strip()]
    for ln in lines:
        if TRASH_REGEX.search(ln):
            quotes = [m.group(2).strip() for m in QUOTE_REGEX.finditer(ln)]
            hits.append({'line': ln, 'quotes': quotes})
    confidence = 0
    if hits:
        # 1 point for the hit, +1 bonus for having a quote in the same line
        confidence = sum(1 + (1 if h['quotes'] else 0) for h in hits)
    return {'trash_flag': int(len(hits) > 0), 'hits': hits, 'confidence': confidence}

print('✅ Trash-talk detector ready.')

In [ ]:
# ---
## 8) Team Recognition Module (LOADED FROM CSV)
# ---
# Make sure your teams.csv file is in the same directory as the notebook.

from rapidfuzz import fuzz
from rapidfuzz import process as fuzzy_process

def load_team_variants(file_path: str = 'teams.csv') -> tuple:
    """
    Loads team data from a CSV file into the required dictionary format.

    Returns: (TEAM_VARIANTS dict, ALL_VARIANTS list)
    """
    team_variants = {}

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            # Skip header line
            next(f)
            for line in f:
                # Clean and split the line
                parts = [p.strip() for p in line.split(',') if p.strip()]
                if not parts:
                    continue

                # The first part is the Primary Name
                primary_name = parts[0]
                # All other parts are variants
                variants = set(parts) # Use a set for unique variants

                # Add the Primary Name to the list if it wasn't listed again
                variants.add(primary_name)

                # Map the primary name to the list of all variants
                team_variants[primary_name] = list(variants)

    except FileNotFoundError:
        print(f"❌ ERROR: Team data file '{file_path}' not found.")
        print("Please create this file and populate it with team names.")
        return {}, []

    # Create the consolidated list of all individual variant names
    all_variants = [variant for sublist in team_variants.values() for variant in sublist]

    return team_variants, all_variants

# Load the data here
TEAM_VARIANTS, ALL_VARIANTS = load_team_variants()

if not TEAM_VARIANTS:
    # Set placeholders if the file was not loaded to prevent errors in extract_teams
    TEAM_VARIANTS = {'Placeholder': ['Placeholder']}
    ALL_VARIANTS = ['Placeholder']

# The team extraction logic remains the same
def extract_teams(text: str, team_variants: dict = TEAM_VARIANTS) -> list:
    """Uses fuzzy matching to identify teams mentioned in the article text."""
    found_teams = set()

    # 1. Look for Exact/Near-Exact Matches (Case-Insensitive)
    text_upper = text.upper()

    for primary_team, variants in team_variants.items():
        # Check if any variant is present in the text
        if any(v.upper() in text_upper for v in variants):
            found_teams.add(primary_team)

    # 2. Use Fuzzy Matching on all variants to catch tricky names/misspellings
    if not found_teams and text:
        text_snippet = text[:2000]

        # Use token set ratio for matching, which ignores word order and noise
        matches = fuzzy_process.extractBests(
            query=text_snippet,
            choices=ALL_VARIANTS,
            scorer=fuzz.token_set_ratio,
            score_cutoff=95, # High score for high confidence
            limit=5
        )
        # Map fuzzy matches back to primary team names
        for match_variant, score, _ in matches:
             for primary_name, variants in team_variants.items():
                if match_variant in variants:
                    found_teams.add(primary_name)

    return sorted(list(found_teams))

print(f'✅ Team recognition module ready. Loaded {len(TEAM_VARIANTS)} teams.')

In [ ]:
#---
## Validation Test
#---

import pandas as pd
from tqdm import tqdm

print("--- STARTING VALIDATION TEST ---")

# 1. Load the raw data (only the first 1000 rows for speed)
# IMPORTANT: This assumes 'articles_raw.parquet' exists after running Cell 6 at least once.
try:
    # Read the full dataframe, but slice it to the first 1000 rows
    articles_df = pd.read_parquet('articles_raw.parquet')

    # We will only use the first 1000 articles for this quick test
    test_df = articles_df.head(1000).copy()

    print(f"Loaded {len(articles_df)} articles. Testing on {len(test_df)} articles.")

except FileNotFoundError:
    print("ERROR: 'articles_raw.parquet' not found. Please run Cell 6 first to generate data.")
    test_df = None

if test_df is not None:
    # 2. Apply Trash Talk Detection and Team Recognition

    # Apply detection logic
    tqdm.pandas(desc="Detecting Trash Talk")
    test_df['is_trash_talk'] = test_df['article_text'].progress_apply(is_trash_talk_v1)

    # Apply team recognition logic
    tqdm.pandas(desc="Extracting Teams")
    test_df['teams_found'] = test_df['article_text'].progress_apply(extract_teams)

    # 3. Filter the results
    # Show only articles where trash talk was detected
    validation_results = test_df[test_df['is_trash_talk'] == True]

    # 4. Print Summary
    print("-" * 40)
    print(f"✅ Validation Complete! Results on {len(test_df)} articles:")
    print(f"   -> Trash Talk Detected In: {validation_results['is_trash_talk'].sum()} articles.")

    # 5. Display a sample of the matches
    if not validation_results.empty:
        print("\nSAMPLE OF DETECTED ARTICLES (5 rows):")
        display(validation_results[['is_trash_talk', 'teams_found', 'article_url']].head())
    else:
        print("No trash talk detected in the sample. Try adjusting keywords or score cutoff.")

print("--- END VALIDATION TEST ---")

In [ ]:
import os
print(os.listdir('.'))

['.config', 'sample_data']


In [ ]:
# ---
## 9) Label Articles and Export Datasets with Final Caching
# This cell will label ALL articles currently in the cache (from all segments completed so far).
# ---

# Define the paths (Using PROJECT_DIR from Cell 3.5)
LABELED_CACHE_PATH_CSV = os.path.join(PROJECT_DIR, 'articles_trash_talk.csv')
RAW_CACHE_PATH_PARQUET = os.path.join(PROJECT_DIR, 'articles_raw.parquet') # Used as cache in Cell 6

def label_articles(articles_rows: List[Dict]) -> pd.DataFrame:
    """Runs the trash-talk and team-matching logic on all articles."""
    recs = []
    for row in tqdm(articles_rows, desc='labeling'):

        # 1. Detect Trash Talk
        det = detect_trash_talk(row['article_text'])

        # 2. Extract Teams (Requires Cell 8 and teams.csv to be working)
        teams = extract_teams(row['title'] + ' ' + row['article_text'])

        recs.append({
            'url': row['url'],
            'published_utc': row['published_utc'],
            'title': row['title'],
            'text_hash': row['text_hash'],
            'trash_flag': det['trash_flag'],
            'trash_snippets': json.dumps([h['line'] for h in det['hits']], ensure_ascii=False),
            'quotes_in_snippets': json.dumps([q for h in det['hits'] for q in h['quotes']], ensure_ascii=False),
            'confidence': det['confidence'],
            'teams_mentioned': ', '.join(teams)
        })
    return pd.DataFrame(recs)


# --- Final Caching Logic (MODIFIED to use Drive path) ---
if os.path.exists(LABELED_CACHE_PATH_CSV):
    # If the final CSV exists, load it directly
    print(f"✅ Final labels found. Loading data from cache: {LABELED_CACHE_PATH_CSV}")
    labeled = pd.read_csv(LABELED_CACHE_PATH_CSV)

else:
    # Check if there are articles to label (from Cell 6)
    if 'articles' not in globals() or not articles:
        # If the variable is not set or empty, try to load the full cache from Cell 6
        if os.path.exists(RAW_CACHE_PATH_PARQUET):
             print("⚠️ 'articles' variable not set, but raw cache found. Loading all raw articles for labeling.")
             df_articles = pd.read_parquet(RAW_CACHE_PATH_PARQUET)
             articles = df_articles.to_dict('records')
        else:
             print("⚠️ No articles found or fetched. Skipping labeling.")
             labeled = pd.DataFrame()
             articles = [] # Ensure articles is defined as an empty list

    if articles: # Only proceed if there are articles to label
        print("⏳ Final cache not found. Running full labeling and analysis on articles...")
        labeled = label_articles(articles)

        # Save the raw article data (used as the cache for Cell 6)
        df_articles = pd.DataFrame(articles)
        df_articles.to_parquet(RAW_CACHE_PATH_PARQUET, index=False)

        # Export the final labeled CSV
        if not labeled.empty:
            labeled.to_csv(LABELED_CACHE_PATH_CSV, index=False)

# --- Audit and Summary ---
print('\n✅ Export Summary:')
print(f' - Raw article cache saved/updated at: {RAW_CACHE_PATH_PARQUET}')
print(f' - Final labeled data saved/loaded from: {LABELED_CACHE_PATH_CSV}')
print(f" - Labeled DataFrame shape: {labeled.shape}")

In [ ]:
# ---
## 9) Label Articles and Export Datasets with Final Caching
# This cell will label ALL articles currently in the cache (from all segments completed so far).
# ---

# Define the paths
LABELED_CACHE_PATH_CSV = os.path.join(OUTPUT_DIR, 'articles_trash_talk.csv')
RAW_CACHE_PATH_PARQUET = os.path.join(OUTPUT_DIR, 'articles_raw.parquet') # Used as cache in Cell 6

def label_articles(articles_rows: List[Dict]) -> pd.DataFrame:
    """Runs the trash-talk and team-matching logic on all articles."""
    recs = []
    for row in tqdm(articles_rows, desc='labeling'):

        # 1. Detect Trash Talk
        det = detect_trash_talk(row['article_text'])

        # 2. Extract Teams (Requires Cell 8 and teams.csv to be working)
        # Note: If you want to force re-labeling after changing keywords or teams.csv,
        # you must manually delete the LABELED_CACHE_PATH_CSV file.
        teams = extract_teams(row['title'] + ' ' + row['article_text'])

        recs.append({
            'url': row['url'],
            'published_utc': row['published_utc'],
            'title': row['title'],
            'text_hash': row['text_hash'],
            'trash_flag': det['trash_flag'],
            'trash_snippets': json.dumps([h['line'] for h in det['hits']], ensure_ascii=False),
            'quotes_in_snippets': json.dumps([q for h in det['hits'] for q in h['quotes']], ensure_ascii=False),
            'confidence': det['confidence'],
            'teams_mentioned': ', '.join(teams)
        })
    return pd.DataFrame(recs)


# --- Final Caching Logic ---
if os.path.exists(LABELED_CACHE_PATH_CSV):
    # If the final CSV exists, load it directly
    print(f"✅ Final labels found. Loading data from cache: {LABELED_CACHE_PATH_CSV}")
    labeled = pd.read_csv(LABELED_CACHE_PATH_CSV)

else:
    # Check if there are articles to label (from Cell 6)
    if 'articles' not in globals() or not articles:
        # If the variable is not set or empty, try to load the full cache from Cell 6
        if os.path.exists(RAW_CACHE_PATH_PARQUET):
             print("⚠️ 'articles' variable not set, but raw cache found. Loading all raw articles for labeling.")
             df_articles = pd.read_parquet(RAW_CACHE_PATH_PARQUET)
             articles = df_articles.to_dict('records')
        else:
             print("⚠️ No articles found or fetched. Skipping labeling.")
             labeled = pd.DataFrame()
             articles = [] # Ensure articles is defined as an empty list

    if articles: # Only proceed if there are articles to label
        print("⏳ Final cache not found. Running full labeling and analysis on articles...")
        labeled = label_articles(articles)

        # Save the raw article data (used as the cache for Cell 6)
        df_articles = pd.DataFrame(articles)
        df_articles.to_parquet(RAW_CACHE_PATH_PARQUET, index=False)

        # Export the final labeled CSV
        if not labeled.empty:
            labeled.to_csv(LABELED_CACHE_PATH_CSV, index=False)

# --- Audit and Summary ---
print('\n✅ Export Summary:')
print(f' - Raw article cache saved/updated at: {RAW_CACHE_PATH_PARQUET}')
print(f' - Final labeled data saved/loaded from: {LABELED_CACHE_PATH_CSV}')
print(f" - Labeled DataFrame shape: {labeled.shape}")

In [ ]:

# ---
## 8) Quick Audit Views — Robust Version
# ---

def sample_for_audit(df: pd.DataFrame, n_pos=10, n_neg=10):
    if "trash_flag" not in df.columns:
        print("❌ Column 'trash_flag' not found. Available columns:", df.columns.tolist())
        return pd.DataFrame()
    if df.empty:
        print("❌ DataFrame is empty. Check earlier cells for errors.")
        return pd.DataFrame()
    # Positive and negative samples
    pos = df[df["trash_flag"] == 1].sample(min(n_pos, len(df[df["trash_flag"] == 1])), random_state=42)
    neg = df[df["trash_flag"] == 0].sample(min(n_neg, len(df[df["trash_flag"] == 0])), random_state=42)
    # Handle cases where one or both sets are empty
    samples = []
    if not pos.empty:
        samples.append(pos)
    if not neg.empty:
        samples.append(neg)
    if not samples:
        print("No positive or negative samples to display.")
        return pd.DataFrame()

    return pd.concat(samples, axis=0).sort_values("trash_flag", ascending=False)

print("\n--- Audit Sample ---")
audit = sample_for_audit(labeled, n_pos=10, n_neg=10)

# Show sample audit results
if not audit.empty:
    display(audit[["published_utc", "trash_flag", "confidence", "title", "url", "trash_snippets"]].head(20))
else:
    print("No audit sample produced — check messages above.")

print("discovered:", len(discovered) if 'discovered' in globals() else 'not set')
print("articles:", len(articles) if 'articles' in globals() else 'not set')
try:
    print("labeled shape:", labeled.shape)
except:
    print("labeled not defined")

**Step 2: Game Stats**

In [ ]:
# --- New Step: Install sportsipy for CFB game data ---
pip -qU install sportsipy
print('✅ sportsipy installed.')

# Imports for scraping game data
from sportsipy.ncaaf.boxscore import Boxscore
from sportsipy.ncaaf.schedule import Schedule
from datetime import date
import pandas as pd
import warnings
warnings.filterwarnings('ignore', category=FutureWarning) # Suppress pandas future warnings